In [ ]:
import pandas as pd

df = pd.read_csv("../data/ab_test_data.csv")

print(df.head())
print()
print(df.info())
print()
print(df["group"].value_counts())

In [ ]:
conversion_summary = (
    df.groupby("group")
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
)

conversion_summary["conversion_rate_pct"] = (
    conversion_summary["conversion_rate"] * 100
)

print(conversion_summary)

In [ ]:
control_rate = conversion_summary.loc["control", "conversion_rate"]
treatment_rate = conversion_summary.loc["treatment", "conversion_rate"]

absolute_lift = treatment_rate - control_rate
relative_lift = (treatment_rate - control_rate) / control_rate

print(f"Control Conversion Rate: {control_rate:.2%}")
print(f"Treatment Conversion Rate: {treatment_rate:.2%}")
print(f"Absolute Lift: {absolute_lift:.2%}")
print(f"Relative Lift: {relative_lift:.2%}")

## Hypothesis Test

We want to determine whether the treatment version produces a higher conversion rate than the control version.

**Null Hypothesis (H₀):** The treatment does not improve the conversion rate.

**Alternative Hypothesis (H₁):** The treatment improves the conversion rate.

**Significance Level:** α = 0.05

In [ ]:
import math

control_conversions = conversion_summary.loc["control", "conversions"]
control_users = conversion_summary.loc["control", "users"]

treatment_conversions = conversion_summary.loc["treatment", "conversions"]
treatment_users = conversion_summary.loc["treatment", "users"]

control_rate = control_conversions / control_users
treatment_rate = treatment_conversions / treatment_users

pooled_rate = (
    control_conversions + treatment_conversions
) / (
    control_users + treatment_users
)

standard_error = math.sqrt(
    pooled_rate * (1 - pooled_rate) *
    ((1 / control_users) + (1 / treatment_users))
)

z_stat = (control_rate - treatment_rate) / standard_error

p_value = 0.5 * (1 + math.erf(z_stat / math.sqrt(2)))

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

## Statistical Conclusion

The one-sided two-proportion Z-test produced a p-value of **0.2872**.

Since the p-value is greater than the significance level of **0.05**, we fail to reject the null hypothesis.

Although the treatment group had a slightly higher conversion rate than the control group, the difference was **not statistically significant**.

Based on this experiment alone, there is not enough evidence to conclude that the treatment improved conversion.

In [ ]:
difference = treatment_rate - control_rate

se_difference = math.sqrt(
    (control_rate * (1 - control_rate) / control_users) +
    (treatment_rate * (1 - treatment_rate) / treatment_users)
)

margin_of_error = 1.96 * se_difference

ci_lower = difference - margin_of_error
ci_upper = difference + margin_of_error

print(f"Difference in Conversion Rate: {difference:.4%}")
print(f"95% Confidence Interval: ({ci_lower:.4%}, {ci_upper:.4%})")

## Confidence Interval Interpretation

The estimated difference in conversion rate between the treatment and control groups is **+0.162 percentage points**.

The 95% confidence interval ranges from **-0.403% to +0.727%**.

Because the confidence interval includes zero, the experiment does not provide strong evidence that the treatment caused an improvement in conversion.

More data or a larger treatment effect may be needed to detect a statistically significant difference.

In [ ]:
import matplotlib.pyplot as plt

rates = conversion_summary["conversion_rate_pct"]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    rates.index,
    rates.values
)

plt.title("A/B Test Conversion Rate")
plt.xlabel("Experiment Group")
plt.ylabel("Conversion Rate (%)")

for bar, value in zip(bars, rates.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{value:.2f}%",
        ha="center"
    )

plt.ylim(0, max(rates.values) + 2)
plt.savefig("../images/conversion_rate_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import math

baseline_rate = control_rate
minimum_detectable_effect = 0.01  # 1 percentage point improvement

expected_treatment_rate = baseline_rate + minimum_detectable_effect

z_alpha = 1.645   # one-sided test at 5%
z_beta = 0.842    # 80% statistical power

pooled_rate = (baseline_rate + expected_treatment_rate) / 2

required_sample_size = (
    (
        z_alpha * math.sqrt(2 * pooled_rate * (1 - pooled_rate))
        + z_beta * math.sqrt(
            baseline_rate * (1 - baseline_rate)
            + expected_treatment_rate * (1 - expected_treatment_rate)
        )
    ) ** 2
    / (minimum_detectable_effect ** 2)
)

print(f"Baseline Conversion Rate: {baseline_rate:.2%}")
print(f"Minimum Detectable Effect: {minimum_detectable_effect:.2%}")
print(f"Required Users Per Group: {math.ceil(required_sample_size):,}")
print(f"Required Total Users: {math.ceil(required_sample_size) * 2:,}")

## Experiment Power Analysis

Using a baseline conversion rate of **11.71%**, an 80% statistical power target, and a significance level of 5%, the experiment would require approximately **13,257 users per group** to detect a **1 percentage-point improvement**.

The experiment included approximately **25,000 users per group**, exceeding this requirement.

However, the observed improvement was only **0.162 percentage points**, substantially smaller than the 1 percentage-point minimum detectable effect used in the sample-size calculation.

This helps explain why the experiment did not produce a statistically significant result despite having a large sample.

In [ ]:
device_analysis = (
    df.groupby(["device", "group"])
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

device_analysis["conversion_rate_pct"] = (
    device_analysis["conversion_rate"] * 100
)

print(device_analysis)

In [ ]:
device_pivot = device_analysis.pivot(
    index="device",
    columns="group",
    values="conversion_rate"
)

device_pivot["absolute_lift"] = (
    device_pivot["treatment"] - device_pivot["control"]
)

device_pivot["relative_lift"] = (
    device_pivot["absolute_lift"] / device_pivot["control"]
)

device_pivot["control_pct"] = device_pivot["control"] * 100
device_pivot["treatment_pct"] = device_pivot["treatment"] * 100
device_pivot["absolute_lift_pct"] = device_pivot["absolute_lift"] * 100
device_pivot["relative_lift_pct"] = device_pivot["relative_lift"] * 100

print(
    device_pivot[
        [
            "control_pct",
            "treatment_pct",
            "absolute_lift_pct",
            "relative_lift_pct"
        ]
    ].round(2)
)

In [ ]:
device_chart = device_analysis.pivot(
    index="device",
    columns="group",
    values="conversion_rate_pct"
)

ax = device_chart.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Conversion Rate by Device and Experiment Group")
plt.xlabel("Device")
plt.ylabel("Conversion Rate (%)")
plt.xticks(rotation=0)
plt.legend(title="Group")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)

plt.tight_layout()

plt.savefig(
    "../images/device_conversion_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import math

device_results = []

for device in df["device"].unique():

    segment = df[df["device"] == device]

    control = segment[segment["group"] == "control"]["converted"]
    treatment = segment[segment["group"] == "treatment"]["converted"]

    control_users = len(control)
    treatment_users = len(treatment)

    control_conversions = control.sum()
    treatment_conversions = treatment.sum()

    control_rate = control.mean()
    treatment_rate = treatment.mean()

    pooled_rate = (
        control_conversions + treatment_conversions
    ) / (
        control_users + treatment_users
    )

    standard_error = math.sqrt(
        pooled_rate * (1 - pooled_rate) *
        ((1 / control_users) + (1 / treatment_users))
    )

    z_stat = (control_rate - treatment_rate) / standard_error

    p_value = 0.5 * (
        1 + math.erf(z_stat / math.sqrt(2))
    )

    device_results.append({
        "device": device,
        "control_rate": control_rate * 100,
        "treatment_rate": treatment_rate * 100,
        "lift_pp": (treatment_rate - control_rate) * 100,
        "p_value": p_value
    })

device_test_results = pd.DataFrame(device_results)

print(device_test_results.round(4))

## Device Segment Findings

The treatment produced higher observed conversion rates across all three device types.

- **Desktop:** Conversion increased from 14.31% to 15.32%, a +1.01 percentage-point lift. The result was statistically significant (p = 0.0358).
- **Mobile:** Conversion increased from 10.51% to 10.97%, but the result was not statistically significant (p = 0.0990).
- **Tablet:** Conversion increased from 11.20% to 11.80%, but the result was not statistically significant (p = 0.2788).

These results suggest that the strongest evidence of a positive treatment effect appears among desktop users. The mobile and tablet results remain inconclusive and would require additional evidence before drawing conclusions about those segments.

In [ ]:
traffic_analysis = (
    df.groupby(["traffic_source", "group"])
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

traffic_analysis["conversion_rate_pct"] = (
    traffic_analysis["conversion_rate"] * 100
)

print(traffic_analysis)

In [ ]:
traffic_pivot = traffic_analysis.pivot(
    index="traffic_source",
    columns="group",
    values="conversion_rate"
)

traffic_pivot["absolute_lift"] = (
    traffic_pivot["treatment"] - traffic_pivot["control"]
)

traffic_pivot["relative_lift"] = (
    traffic_pivot["absolute_lift"] / traffic_pivot["control"]
)

traffic_pivot["control_pct"] = traffic_pivot["control"] * 100
traffic_pivot["treatment_pct"] = traffic_pivot["treatment"] * 100
traffic_pivot["absolute_lift_pct"] = traffic_pivot["absolute_lift"] * 100
traffic_pivot["relative_lift_pct"] = traffic_pivot["relative_lift"] * 100

print(
    traffic_pivot[
        [
            "control_pct",
            "treatment_pct",
            "absolute_lift_pct",
            "relative_lift_pct"
        ]
    ].round(2)
)

In [ ]:
traffic_results = []

for source in df["traffic_source"].unique():

    segment = df[df["traffic_source"] == source]

    control = segment[segment["group"] == "control"]["converted"]
    treatment = segment[segment["group"] == "treatment"]["converted"]

    control_users = len(control)
    treatment_users = len(treatment)

    control_conversions = control.sum()
    treatment_conversions = treatment.sum()

    control_rate = control.mean()
    treatment_rate = treatment.mean()

    pooled_rate = (
        control_conversions + treatment_conversions
    ) / (
        control_users + treatment_users
    )

    standard_error = math.sqrt(
        pooled_rate * (1 - pooled_rate) *
        ((1 / control_users) + (1 / treatment_users))
    )

    z_stat = (control_rate - treatment_rate) / standard_error

    p_value = 0.5 * (
        1 + math.erf(z_stat / math.sqrt(2))
    )

    traffic_results.append({
        "traffic_source": source,
        "control_rate": control_rate * 100,
        "treatment_rate": treatment_rate * 100,
        "lift_pp": (treatment_rate - control_rate) * 100,
        "p_value": p_value
    })

traffic_test_results = pd.DataFrame(traffic_results)

print(traffic_test_results.round(4))

## Traffic Source Findings

Treatment performance varied across acquisition channels.

- **Email:** Conversion increased from 11.67% to 12.88%, a +1.21 percentage-point lift. The result was statistically significant (p = 0.0322).
- **Social:** Conversion increased by +1.03 percentage points, but the result was not statistically significant (p = 0.0842).
- **Paid:** Conversion increased by +0.59 percentage points, but the result was not statistically significant (p = 0.1332).
- **Organic:** Conversion increased by only +0.06 percentage points and was not statistically significant (p = 0.4503).

Among the traffic-source segments, email users showed the strongest statistical evidence of a positive treatment effect. The remaining channel results are inconclusive.

In [ ]:
traffic_chart = traffic_analysis.pivot(
    index="traffic_source",
    columns="group",
    values="conversion_rate_pct"
)

ax = traffic_chart.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Conversion Rate by Traffic Source and Experiment Group")
plt.xlabel("Traffic Source")
plt.ylabel("Conversion Rate (%)")
plt.xticks(rotation=0)
plt.legend(title="Group")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)

plt.tight_layout()

plt.savefig(
    "../images/traffic_source_conversion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()